In [11]:
import conllu # our id data not just a plain text its complex table format where every word has 10+ columns of language info. This lib saves us from writing a confusing Regex parser to read it
import numpy as np
from collections import defaultdict

In [12]:
def load_conllu_data(file_path):
    data_list = [] # this is empty list but in the end it will hold our cleaned data

    with open(file_path, 'r', encoding='utf-8') as f: # openning a file in a readable mod with the utf-8 encoding 

        for sentence in conllu.parse_incr(f): # this is read file sentence by sentence instead of loading all at once into ram
            words = []
            tags = []
            for token in sentence: # we will loop through every token
                words.append(token['form']) # actual word seen in text
                tags.append(token['upos']) # universal part of speech tag    
            data_list.append((words, tags)) # this row func easier to show ["makan", "nasi"], ["verb", "noun"]
    return data_list

In [13]:
train_data = load_conllu_data('id_gsd-ud-train.conllu')
dev_data = load_conllu_data('id_gsd-ud-dev.conllu')
test_data = load_conllu_data('id_gsd-ud-test.conllu')

print(f"Training sentences: {len(train_data)}")
print(f"Dev sentences: {len(dev_data)}")
print(f"Test sentences: {len(test_data)}")
print(f"Sample sentence: {train_data[0]}")

Training sentences: 4477
Dev sentences: 559
Test sentences: 557
Sample sentence: (['Sembungan', 'adalah', 'sebuah', 'desa', 'yang', 'terletak', 'di', 'kecamatan', 'Kejajar', ',', 'kabupaten', 'Wonosobo', ',', 'Jawa', 'Tengah', ',', 'Indonesia', '.'], ['PROPN', 'AUX', 'DET', 'NOUN', 'PRON', 'VERB', 'ADP', 'NOUN', 'PROPN', 'PUNCT', 'NOUN', 'PROPN', 'PUNCT', 'PROPN', 'PROPN', 'PUNCT', 'PROPN', 'PUNCT'])


In [14]:
class HMM: # hmm have 2 parts training and decoding
    def __init__(self):
        self.transitions = defaultdict(lambda: defaultdict(int)) # store matrix of tag to tag
        self.emissions = defaultdict(lambda: defaultdict(int)) # store matrix of tag to word

        self.tags = set() 
        self.vocab = set() # these two will hold list of uniqe tags and words during train

    def train(self, data):
        print("Training HMM")
        for words, tags in data:
            prev_tag = "<START>"
            self.tags.add(prev_tag) # with this we can assume every sentences start with a start state with this we can learn how usually sentences start
            
            for word, tag in zip(words, tags): # zip will let us loop word and tag continuesly
                self.transitions[prev_tag][tag] += 1 # prev foolowed tag give +
                self.emissions[tag][word] += 1 # tag followed word give +

                self.tags.add(tag) 
                self.vocab.add(word) # wee keep known tag and words

                prev_tag = tag # update it so nextiteration what came before 
            
            self.transitions[prev_tag]["<END>"] += 1 # and at the end of the loop we will tag last + 1 as end 


        self.trans_probs = defaultdict(lambda: 1e-10)
        self.emit_probs = defaultdict(lambda: 1e-10) # this is called smoothing it will prevent math error
        
        for prev, next_dict in self.transitions.items():
            total = sum(next_dict.values())
            for tag, count in next_dict.items():
                self.trans_probs[(prev, tag)] = count / total 
                
        for tag, word_dict in self.emissions.items():
            total = sum(word_dict.values())
            for word, count in word_dict.items():
                self.emit_probs[(tag, word)] = count / total
        print("Training Complete.") # upper 2 block will convert transition and emission count to probalities

    def predict(self, sentence):
        V = [{}] # we will is viterbi it will store best probability at each step
        path = {} # it will store actual path past of tags
        
        for tag in self.tags: # first word
            if tag in ["<START>", "<END>"]: continue
            V[0][tag] = self.trans_probs[("<START>", tag)] * self.emit_probs[(tag, sentence[0])]
            path[tag] = [tag]
            
        for t in range(1, len(sentence)): # rest of it 
            V.append({})
            new_path = {}
            
            for curr_tag in self.tags: # it will look out for every possible current tag and continue with max prob it will use Previous_Score * Transition_Prob * Emission_Prob
                if curr_tag in ["<START>", "<END>"]: continue
                (prob, best_prev) = max(
                    (V[t-1][prev] * self.trans_probs[(prev, curr_tag)] * self.emit_probs[(curr_tag, sentence[t])], prev) 
                    for prev in self.tags if prev in V[t-1]
                )

                V[t][curr_tag] = prob # we will store the highest one 
                new_path[curr_tag] = path[best_prev] + [curr_tag]
            path = new_path # and extend th epath
            
        (prob, best_final) = max((V[len(sentence)-1][tag] * self.trans_probs[(tag, "<END>")], tag) for tag in self.tags if tag in V[len(sentence)-1]) # Multiply by the probability of transitioning to end
        return path[best_final]

In [15]:
hmm = HMM() # lets create an object 
hmm.train(train_data) # than train it

correct = 0
total = 0
print("Evaluating HMM on first 100 test sentences...")
for words, true_tags in test_data[:100]: # lets look accuracy on firsst 100 row 
    try:
        pred_tags = hmm.predict(words)

        for p, t in zip(pred_tags, true_tags):# compare part
            if p == t: correct += 1
            total += 1
    except: pass # we will skip sentences cause error

print(f"HMM Accuracy: {correct/total:.2%}")

Training HMM
Training Complete.
Evaluating HMM on first 100 test sentences...
HMM Accuracy: 88.36%
